# M2 Displacement T4 재실행
고정 revision c5d09511ac1c7a88156440e861a4719f85b6510a. 실제 실험은 완료했으며 재실행은 선택 사항입니다.
T4 GPU를 선택하고 위에서부터 실행합니다. 입력 셀에서 개인 M2_verified_148_input.zip을 업로드합니다.
기존 결과 파일을 재사용/덮어쓰지 않습니다. private notebook에는 입력이 포함되어 별도 업로드가 필요 없습니다.


In [ ]:
import torch, sys
print(sys.version, torch.__version__)
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0), 'T4 required; stop'
print('GPU', torch.cuda.get_device_name(0))


In [ ]:
import hashlib, json, subprocess, sys, uuid
from pathlib import Path
from google.colab import files
BASE = Path('/content') / ('m2_displacement_' + uuid.uuid4().hex)
BASE.mkdir()
ROOT = BASE/'repo'
subprocess.run(['git','clone','-q','--branch','experiment/m2-displacement-loss-20261116','https://github.com/darkkyoung/REWIND-like-prototype.git',str(ROOT)],check=True)
subprocess.run(['git','checkout','--detach','c5d09511ac1c7a88156440e861a4719f85b6510a'],cwd=ROOT,check=True)
uploaded = files.upload()
assert len(uploaded)==1, 'Upload exactly M2_verified_148_input.zip'
raw = next(iter(uploaded.values()))
assert hashlib.sha256(raw).hexdigest() == '184967a9732702fc0459350974b87e67f6499144dcfdbec48b18cb657805d74d'
INPUT = BASE/'M2_verified_148_input.zip'
INPUT.write_bytes(raw)
print('PASS: verified input bundle')


In [ ]:
# CPU 구현 검사 및 기존 회귀 검사. 실제 학습은 다음 셀만 수행합니다.
TEST_LOG = BASE/'validation_tests.log'
commands = [
    [sys.executable,'-m','unittest','experiments.final_m2.displacement_loss_20261116.test_displacement','-v'],
    [sys.executable,'-m','unittest','discover','-s','tests','-p','test_m2.py','-v'],
    [sys.executable,'-m','unittest','discover','-s','tests','-p','test_final_metrics.py','-v'],
    [sys.executable,'-m','unittest','discover','-s','experiments/final_m2/diagnosis_20261111','-p','test_diagnose.py','-v']]
with TEST_LOG.open('x') as log:
    for cmd in commands:
        result = subprocess.run(cmd,cwd=ROOT,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
        log.write(result.stdout); log.flush(); print(result.stdout)
        assert result.returncode == 0, 'Validation failed: do not train'


In [ ]:
# 정확히 두 군 각100 update. 기존 결과 경로 재사용/덮어쓰기 금지.
RESULT = BASE/'t4_displacement_result'
assert not RESULT.exists(), 'Do not rerun training into an existing result'
TRAIN_LOG = BASE/'execution.log'
cmd = [sys.executable,'experiments/final_m2/displacement_loss_20261116/run_experiment.py',
       '--input-bundle',str(INPUT),'--output',str(RESULT)]
with TRAIN_LOG.open('x') as log:
    proc = subprocess.Popen(cmd,cwd=ROOT,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=1)
    for line in proc.stdout:
        log.write(line);log.flush();print(line,end='',flush=True)
    code = proc.wait()
print('PROCESS_EXIT',code)
assert code == 0, 'Preserve failure.json and execution.log; do not tune or repeat automatically'
evaluation = json.loads((RESULT/'evaluation.json').read_text())
print(json.dumps({'execution_status':evaluation['execution_status'],'quality':evaluation['quality'],'results':evaluation['results']},indent=2))


In [ ]:
# 학습은 재실행하지 않고 저장된 실제 배열만 시각화합니다.
FIGURES = BASE/'figures'
subprocess.run([sys.executable,'experiments/final_m2/displacement_loss_20261116/plot_results.py',
                '--input',str(RESULT),'--output',str(FIGURES)],cwd=ROOT,check=True)
from IPython.display import display, Image
for name in ['frame_displacement_comparison.png','skeleton_visualization.png','root_relative_comparison.png']:
    display(Image(filename=str(FIGURES/name)))


In [ ]:
# 성공/실패 수치·checkpoint·예측·로그를 그대로 보관합니다.
import zipfile, hashlib
RESULT_ZIP = BASE/'REWIND_M2_Displacement_T4_results.zip'
with zipfile.ZipFile(RESULT_ZIP,'x',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RESULT.rglob('*')):
        if p.is_file() and 'private_input' not in p.parts:z.write(p,'t4/'+str(p.relative_to(RESULT)))
    for p in sorted(FIGURES.glob('*')):
        if p.is_file():z.write(p,'figures/'+p.name)
    for p in [TEST_LOG,TRAIN_LOG]:z.write(p,p.name)
print('RESULT_SHA256',hashlib.sha256(RESULT_ZIP.read_bytes()).hexdigest())
from google.colab import files
files.download(str(RESULT_ZIP))
